# Import Depedencies

In [19]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

warnings.filterwarnings('ignore')

print('Setup Selesai')

Setup Selesai


# Load Data

In [11]:
def load_data(path):
    for dirname, _, filenames in os.walk('../data'):
        if path in filenames:
            return os.path.join(dirname, path)

df = pd.read_csv(load_data('dataset_analisis_pantai_kenjeran.csv'))
df['Transect_ID'] = df['Transect_ID'].astype('category')
df['DATE_'] = pd.to_datetime(df['DATE_'])
df[['Lag_1', 'Lag_2', 'Lag_3']] = df[['Lag_1', 'Lag_2', 'Lag_3']].fillna(0)
df

,Transect_ID,Tahun,DATE_,Shoreline_Distance,Lag_1,Lag_2,Lag_3,Longitude_UTM,Latitude_UTM,azimuth,tcd,lrr,epr,nsm,sce,shrcount
0,2,2015,2015-01-01,175.033770,0.000000,0.000000,0.000000,697516.355020,9.201540e+06,73.41,50.0,0.002941,0.006445,0.064449,0.064509,11
1,2,2016,2016-01-01,175.098218,175.033770,0.000000,0.000000,697516.441562,9.201540e+06,73.41,50.0,0.002941,0.006445,0.064449,0.064509,11
2,2,2017,2017-01-01,175.098007,175.098218,175.033770,0.000000,697516.441774,9.201540e+06,73.41,50.0,0.002941,0.006445,0.064449,0.064509,11
3,2,2018,2018-01-01,175.097883,175.098007,175.098218,175.033770,697516.441898,9.201540e+06,73.41,50.0,0.002941,0.006445,0.064449,0.064509,11
4,2,2019,2019-01-01,175.098278,175.097883,175.098007,175.098218,697516.441502,9.201540e+06,73.41,50.0,0.002941,0.006445,0.064449,0.064509,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1126,107,2019,2019-01-01,105.064905,139.567095,70.562714,1.558678,699302.176292,9.198330e+06,29.60,5300.0,20.747377,25.300851,253.008509,253.008509,8
1127,107,2020,2020-01-01,185.569778,105.064905,139.567095,70.562714,699341.938472,9.198400e+06,29.60,5300.0,20.747377,25.300851,253.008509,253.008509,8
1128,107,2021,2021-01-01,139.567011,185.569778,105.064905,139.567095,699319.217235,9.198360e+06,29.60,5300.0,20.747377,25.300851,253.008509,253.008509,8
1129,107,2022,2022-01-01,139.567040,139.567011,185.569778,105.064905,699319.217249,9.198360e+06,29.60,5300.0,20.747377,25.300851,253.008509,253.008509,8


# Feature Engineering

## Feature Creation

In [12]:
# Segmen
tcd_min = df['tcd'].min()
tcd_max = df['tcd'].max()
transect_unique = df['Transect_ID'].nunique()
amount_class = int(np.ceil(1 + 3.3 * np.log10(transect_unique)))
def segmen(tcd):
    diff = tcd_max - tcd_min
    k = diff / amount_class
    return int((tcd - tcd_min) // (k)) + 1
df['segmen'] = df['tcd'].apply(segmen).astype(str)
df['segmen'] = df['segmen'].clip(upper=str(amount_class))
df['segmen'] = df['segmen'].astype('category')

# Temporal Features
df['year_index'] = df['Tahun'] - df['Tahun'].min()
df['change_prev_year'] = df['Shoreline_Distance'] - df['Lag_1']
df['change_2_year'] = df['Lag_1'] - df['Lag_2']
df['change_3_year'] = df['Lag_2'] - df['Lag_3']
df['velocity_recent'] = (df['Shoreline_Distance'] - df['Lag_2']) / 2
df['acceleration_change'] = (df['Lag_1'] - df['Lag_2']) - (df['Lag_2'] - df['Lag_3'])

# Lag Statistik
df['lag_mean_3'] = df[['Lag_1', 'Lag_2', 'Lag_3']].mean(axis=1)
df['lag_std_3'] = df[['Lag_1', 'Lag_2', 'Lag_3']].std(axis=1)
df['lag_min_3'] = df[['Lag_1', 'Lag_2', 'Lag_3']].min(axis=1)
df['lag_max_3'] = df[['Lag_1', 'Lag_2', 'Lag_3']].max(axis=1)
df['lag_range_3'] = df['lag_max_3'] - df['lag_min_3']

# Spatial Features
df['relative_longitude'] = df['Longitude_UTM'] - df['Longitude_UTM'].min()
df['relative_latitude'] = df['Latitude_UTM'] - df['Latitude_UTM'].min()
df['distance_from_origin'] = np.sqrt(
    df['relative_longitude']**2 + df['relative_latitude']**2
)
df['transect_rank_west_east'] = df['Longitude_UTM'].rank(method='dense', ascending=True).astype(int)
df['coastal_zone'] = pd.qcut(df['Longitude_UTM'], q=5, labels=['Zona1', 'Zona2', 'Zona3', 'Zona4', 'Zona5'])
df['coastal_zone'] = df['coastal_zone'].astype('category')

# Azimuth Features
rad = np.pi / 180
df['azimuth_sin'] = np.sin(df['azimuth'] * rad)
df['azimuth_cos'] = np.cos(df['azimuth'] * rad)
df['azimuth_deviation'] = df['azimuth'] - df['azimuth'].mean()
bins = [0, 22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5, 360]
labels = [
    'Utara', 'Timur Laut', 'Timur', 'Tenggara',
    'Selatan', 'Barat Daya', 'Barat', 'Barat Laut', 'Utara_360'
]
df['orientation_class'] = pd.cut(
    df['azimuth'], bins=bins, 
    labels=labels, include_lowest=True, right=False
)
df['orientation_class'] = df['orientation_class'].replace('Utara_360', 'Utara')
df['orientation_class'] = df['orientation_class'].astype('category')

# Interaction Features
df['lrr_sce_ratio'] = df['lrr'] / (df['sce'] + 1)
df['epr_nsm_ratio'] = df['epr'] / (abs(df['nsm']) + 1)
df['shoreline_tcd_ratio'] = df['Shoreline_Distance'] / df['tcd']
df['shoreline_density'] = df['Shoreline_Distance'] / df['shrcount']
df['lrr_epr_diff'] = abs(df['lrr'] - df['epr'])

## Feature Selection

In [ ]:
target = 'Shoreline_Distance'
not_cols = ['Tahun', target, 'shrcount']
cat_cols = df.select_dtypes(include='category').columns
num_cols = [i for i in df.select_dtypes(include=['int64', 'float64']).columns if i not in not_cols]

X = df[num_cols + ['DATE_'] + cat_cols].copy()
y = df[target].copy()

X = X.sort_values('DATE_')
y = y.loc[X.index]

# Split Data
split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()